In [16]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from category_encoders import BinaryEncoder
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

from sklearn.ensemble import RandomForestRegressor

from sklearn.feature_selection import (
    mutual_info_classif,
    chi2,
    f_classif,
    SelectKBest
)
from sklearn.metrics import roc_auc_score, classification_report

In [2]:
df_real_estate = pd.read_csv('Master_RealEstate_Combined_v1.csv')
df = df_real_estate.copy()
df.sample(5)

,Type,Address,Price,Price_Per_M2,Area,Rooms,Bathrooms,Floor,Finish,View,Amenities,Amenities_Count,Is_Installments
5524,Apartment,Rehab,4300000.0,47777.777778,90.0,2.0,1.0,First Floor,1,Other,NaN,0,0
4506,Penthouse,Other,16000000.0,NaN,NaN,3.0,NaN,Last Floor/Roof,0,Other,NaN,0,0
2251,Apartment,cairo al oubour el hay el tamen,4000000.0,19900.000000,201.0,3.0,2.0,1.0,1,Main Street,"Elevator, Security, Garden, Balcony",4,1
373,Studio,cairo new cairo 6th settlement compounds golf ...,2800000.0,53846.000000,52.0,1.0,1.0,1.0,1,Garden,Garden,1,1
7104,Commercial,New Cairo,10000000.0,NaN,NaN,NaN,NaN,Not Mentioned,1,Other,NaN,0,0


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8865 entries, 0 to 8864
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Type             8865 non-null   object 
 1   Address          8865 non-null   object 
 2   Price            8864 non-null   float64
 3   Price_Per_M2     5846 non-null   float64
 4   Area             5792 non-null   float64
 5   Rooms            6919 non-null   float64
 6   Bathrooms        7018 non-null   float64
 7   Floor            8276 non-null   object 
 8   Finish           8865 non-null   int64  
 9   View             8839 non-null   object 
 10  Amenities        5405 non-null   object 
 11  Amenities_Count  8865 non-null   int64  
 12  Is_Installments  8865 non-null   int64  
dtypes: float64(5), int64(3), object(5)
memory usage: 900.5+ KB


In [4]:
df.describe()

d:\Apps\anaconda\envs\practiceml\Lib\site-packages\pandas\core\nanops.py:1016: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


,Price,Price_Per_M2,Area,Rooms,Bathrooms,Finish,Amenities_Count,Is_Installments
count,8.864000e+03,5.846000e+03,5792.000000,6.919000e+03,7018.000000,8865.000000,8865.000000,8865.000000
mean,8.931347e+06,inf,235.296961,1.453384e+08,2.744514,0.950818,1.372589,0.434856
std,1.303270e+07,NaN,1468.082564,5.404888e+09,7.079778,0.216260,1.597618,0.495766
min,3.500000e+03,2.000000e+01,0.000000,1.000000e+00,1.000000,0.000000,0.000000,0.000000
25%,4.340598e+06,2.709975e+04,120.000000,2.000000e+00,2.000000,1.000000,0.000000,0.000000
50%,6.621527e+06,4.357100e+04,157.500000,3.000000e+00,2.000000,1.000000,1.000000,0.000000
75%,1.020000e+07,6.403981e+04,200.000000,3.000000e+00,3.000000,1.000000,2.000000,1.000000
max,5.999490e+08,inf,63000.000000,2.011171e+11,180.000000,1.000000,7.000000,1.000000


`Dropping Price_Per_M2 column because nearly the half is missing or infinite value and it introduces multicolinearity`

In [5]:
df = df.drop(['Price_Per_M2'], axis=1)

In [6]:
df['Floor'].unique()

array([nan, '2.0', '0.0', '3.0', '11.0', '7.0', '1.0', '4.0', '9.0',
       '5.0', '6.0', '10.0', '13.0', '12.0', '18.0', '8.0', '45.0',
       '15.0', '14.0', '25.0', '17.0', '16.0', 'Not Mentioned',
       'Ground/Basement', 'First Floor', 'Last Floor/Roof',
       'Typical Floor'], dtype=object)

`Floor column have overlapped values and Typical Floor and Last Floor/Roof holds a wide range of possible values, so to make data consistent we need need to categorize floors into bins`

In [7]:
df['Floor'] = df['Floor'].replace('Not Mentioned', np.nan)

In [8]:
floor_bins = {
    'First Floors/Ground': ['First Floor', 'Ground/Basement', '0.0', '1.0', '2.0', '3.0', '4.0'],
    'Typical Floors': ['Typical Floor', '5.0', '6.0', '7.0', '8.0'],
    'High Floors': ['9.0', '10.0', '11.0', '12.0', '13.0', '14.0', '15.0'],
    'Last Floors/Roof': ['16.0', '17.0', '18.0', '25.0', '45.0', 'Last Floor/Roof']
}

reverse_map = {val: category for category, values in floor_bins.items() for val in values}
df['Floor'] = df['Floor'].map(reverse_map)
df['Floor'].sample(50)

3875         Typical Floors
2802    First Floors/Ground
1531    First Floors/Ground
2154    First Floors/Ground
6389                    NaN
8642                    NaN
7071       Last Floors/Roof
7167                    NaN
7914                    NaN
1184    First Floors/Ground
7508    First Floors/Ground
267     First Floors/Ground
2362    First Floors/Ground
8339    First Floors/Ground
6928    First Floors/Ground
5648                    NaN
1669    First Floors/Ground
6864    First Floors/Ground
7475    First Floors/Ground
3910    First Floors/Ground
8323                    NaN
1598    First Floors/Ground
397          Typical Floors
6467    First Floors/Ground
2457    First Floors/Ground
2701    First Floors/Ground
872     First Floors/Ground
599     First Floors/Ground
3699                    NaN
2014    First Floors/Ground
7438                    NaN
8000                    NaN
7948                    NaN
119     First Floors/Ground
3678    First Floors/Ground
6453                

`We need to fix the infinite value in the Price_Per_M2 by replacing it with the maximum non-infinite value`

`Handling duplicates`

In [9]:
df.duplicated().sum()

np.int64(31)

In [10]:
df.drop_duplicates(inplace=True)

`Splitting the data before preprocessing to prevent data leakage`

In [11]:
df['Price'].dropna(axis=0)

0        6800000.0
1        5000000.0
2        5500000.0
3        9974100.0
4        1600000.0
           ...    
8860     5450000.0
8861    10250000.0
8862    11500000.0
8863     6000000.0
8864     2600000.0
Name: Price, Length: 8833, dtype: float64

In [18]:
X_train, X_test, y_train, y_test = train_test_split(df.drop('Price', axis=1), df['Price'], test_size=0.33, random_state=42)

`Data transformations`

In [13]:
numeric_features = X_train.select_dtypes('number').columns
categorical_features = X_train.select_dtypes(exclude='number').columns

In [19]:
for col in categorical_features:
    missing_training_mask = X_train[col].isna()
    missing_testing_mask = X_test[col].isna()

    encode = BinaryEncoder(cols =[col], drop_invariant=True)
    X_train_encoded = encode.fit_transform(X_train[[col]])
    X_test_encoded = encode.transform(X_test[[col]])

    X_train_encoded.loc[missing_training_mask, :] = np.nan
    X_test_encoded.loc[missing_testing_mask, :] = np.nan

    X_train = pd.concat([X_train.drop(col, axis=1), X_train_encoded], axis=1)
    X_test = pd.concat([X_test.drop(col, axis=1), X_test_encoded], axis=1)

X_train

,Area,Rooms,Bathrooms,Finish,Amenities_Count,Is_Installments,Type_0,Type_1,Type_2,Type_3,...,View_2,View_3,Amenities_0,Amenities_1,Amenities_2,Amenities_3,Amenities_4,Amenities_5,Amenities_6,Amenities_7
1781,140.0,3.0,2.0,1,2,0,0.0,0.0,0.0,1.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
5635,NaN,3.0,NaN,1,0,0,0.0,0.0,0.0,1.0,...,1.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
303,3300.0,NaN,NaN,1,5,0,0.0,0.0,1.0,0.0,...,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
450,132.0,3.0,2.0,1,0,0,0.0,0.0,0.0,1.0,...,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
784,260.0,3.0,2.0,1,4,0,0.0,0.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5764,168.0,3.0,3.0,1,0,1,0.0,1.0,0.0,0.0,...,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5221,NaN,3.0,3.0,1,0,0,0.0,1.0,1.0,1.0,...,1.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5420,NaN,2.0,3.0,1,0,1,0.0,0.0,0.0,1.0,...,1.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
867,295.0,4.0,3.0,1,1,1,0.0,1.0,1.0,1.0,...,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0


In [20]:
# Select an estimator that automatically handels feature selections and overfitting
impute = IterativeImputer(estimator=RandomForestRegressor(n_estimators=50, random_state=42), tol=0.1, random_state=42)
X_train_processed = impute.fit_transform(X_train)
X_test_processed = impute.transform(X_test)
X_train_processed

array([[1.40000000e+02, 3.00000000e+00, 2.00000000e+00, ...,
        0.00000000e+00, 0.00000000e+00, 1.00000000e+00],
       [1.95577143e+02, 3.00000000e+00, 3.00000000e+00, ...,
        1.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [3.30000000e+03, 1.56800000e+01, 8.28000000e+00, ...,
        0.00000000e+00, 1.00000000e+00, 0.00000000e+00],
       ...,
       [1.39600000e+02, 2.00000000e+00, 3.00000000e+00, ...,
        1.00000000e+00, 3.80000000e-01, 2.00000000e-02],
       [2.95000000e+02, 4.00000000e+00, 3.00000000e+00, ...,
        1.00000000e+00, 1.00000000e+00, 0.00000000e+00],
       [1.91460000e+02, 3.00000000e+00, 2.82000000e+00, ...,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00]], shape=(5918, 36))

`Avoiding scaling encoded features by selecting only numeric indecies to scale`

In [ ]:
scale = StandardScaler()
X_train[:, :3] = scale.fit_transform(X_train_processed[:, :3])
X_test[:, :3] = scale.transform(X_test_processed[:, :3])
X_train

array([[-0.09485359, -0.07931903, -0.12472186, ...,  0.        ,
         0.        ,  1.        ],
       [-0.07689022, -0.07931903, -0.01653149, ...,  1.        ,
         0.        ,  0.        ],
       [ 0.92650601, -0.07931903,  0.55471367, ...,  0.        ,
         1.        ,  0.        ],
       ...,
       [-0.09498288, -0.07931903, -0.01653149, ...,  1.        ,
         0.38      ,  0.02      ],
       [-0.04475526, -0.07931903, -0.01653149, ...,  1.        ,
         1.        ,  0.        ],
       [-0.07822095, -0.07931903, -0.03600576, ...,  0.        ,
         0.        ,  0.        ]], shape=(5918, 36))